In [204]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [205]:
with open('names.txt','r') as f:
    words = f.read().splitlines()

In [206]:
characters = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(characters)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [207]:
import torch

In [208]:
import random
random.seed(42)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

train_words = words[:n1]   
dev_words   = words[n1:n2] 
test_words  = words[n2:] 


In [232]:
import numpy as np

ks=[]
lss=[]
N = torch.zeros((27, 27, 27), dtype=torch.int32)
for w in train_words:
    chrs = ['.', '.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chrs, chrs[1:], chrs[2:]):
        idx1 = stoi[ch1]
        idx2 = stoi[ch2]
        idx3 = stoi[ch3]
        N[idx1,idx2,idx3] +=1

for k in np.arange(0.25, 2.25, 0.25).tolist():
    P = (N + k).float()
    P /= P.sum(dim=2, keepdim=True)

    log_likelihood = 0.0
    n = 0
    for w in dev_words:
        chrs = ['.', '.'] + list(w) + ['.']
        for ch1, ch2, ch3 in zip(chrs, chrs[1:], chrs[2:]):
            prob = P[stoi[ch1], stoi[ch2], stoi[ch3]]
            log_likelihood += torch.log(prob)
            n += 1
            
    dev_loss = -log_likelihood / n
    ks.append(k)
    lss.append(dev_loss)
    print(f'Smoothing (k={k:4.2f}) -> Dev Loss: {dev_loss.item():.4f}')


Smoothing (k=0.25) -> Dev Loss: 2.2227
Smoothing (k=0.50) -> Dev Loss: 2.2267
Smoothing (k=0.75) -> Dev Loss: 2.2316
Smoothing (k=1.00) -> Dev Loss: 2.2365
Smoothing (k=1.25) -> Dev Loss: 2.2413
Smoothing (k=1.50) -> Dev Loss: 2.2461
Smoothing (k=1.75) -> Dev Loss: 2.2508
Smoothing (k=2.00) -> Dev Loss: 2.2554


In [233]:
min_loss = min(lss)
best_k = ks[lss.index(min_loss)]
p = (N + best_k).float()
p /= p.sum(2, keepdim=True)


In [234]:
log_likelihood = 0.0
n = 0
for w in test_words:
    chrs = ['.','.'] + list(w) + ['.']
    for ch1, ch2,ch3 in zip(chrs, chrs[1:],chrs[2:]):
        idx1 = stoi[ch1]
        idx2 = stoi[ch2]
        idx3 = stoi[ch3]
        prob = p[idx1,idx2,idx3]
        log_prob = torch.log(prob)
        log_likelihood += log_prob
        n+=1

test_loss = -log_likelihood / n
print(f"trigram test loss: {test_loss.item():.4f}")
print("bigram test loss: 2.454")


trigram test loss: 2.2238
bigram test loss: 2.454


In [235]:
#sayım tablosu
generator = torch.Generator().manual_seed(42)
for i in range(5):
    out = []
    idx1, idx2 = 0, 0
    while True:
        p_ = p[idx1, idx2]
        idx3 = torch.multinomial(p_, num_samples=1, replacement=True, generator=generator).item()
        out.append(itos[idx3])
        if idx3 == 0:
            break
        idx1, idx2 = idx2, idx3
    print("".join(out))


ye.
syahle.
amen.
leekkim.
mannya.


In [236]:
import torch.nn.functional as F

In [227]:
# dataesti oluştur
xs1,xs2, ys = [], [],[]

for w in words:
  chrs = ['.','.'] + list(w) + ['.']
  for ch1, ch2,ch3 in zip(chrs, chrs[1:],chrs[2:]):
      idx1 = stoi[ch1]
      idx2 = stoi[ch2]
      idx3 = stoi[ch3]
      xs1.append(idx1)
      xs2.append(idx2)
      ys.append(idx3)

xs1 = torch.tensor(xs1)
xs2 = torch.tensor(xs2)
ys = torch.tensor(ys)

num = xs1.nelement() + xs2.nelement()
print('number of examples: ', num)

number of examples:  456292


In [228]:
W = torch.randn((54, 27), generator=generator, requires_grad=True)

In [237]:
# gradient descent

for k in range(100):
  
  # forward pass
  xenc1 = F.one_hot(xs1, num_classes=27).float()
  xenc2 = F.one_hot(xs2, num_classes=27).float()
  xenc = torch.cat([xenc1, xenc2], dim=1)  
  logits = xenc @ W 
  counts = logits.exp() #softmax
  probs = counts / counts.sum(dim=1, keepdims=True) #softmax
  loss = -probs[torch.arange(len(ys)), ys].log().mean() + 0.01*(W**2).mean()#nll loss  
  
  print(loss.item())
  
  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()
  
  # update
  W.data += -10 * W.grad

2.4854626655578613
2.4840478897094727
2.482659101486206
2.481294870376587
2.479954957962036
2.4786384105682373
2.4773452281951904
2.476074695587158
2.4748260974884033
2.4735989570617676
2.4723925590515137
2.4712069034576416
2.470041036605835
2.4688947200775146
2.4677677154541016
2.4666590690612793
2.4655685424804688
2.46449613571167
2.4634408950805664
2.462402582168579
2.461381673812866
2.460376501083374
2.4593873023986816
2.458413600921631
2.4574551582336426
2.456511974334717
2.455583095550537
2.4546687602996826
2.453768491744995
2.4528820514678955
2.4520087242126465
2.451148509979248
2.4503014087677
2.449467182159424
2.4486451148986816
2.4478354454040527
2.4470372200012207
2.446251392364502
2.445476531982422
2.4447124004364014
2.4439597129821777
2.4432177543640137
2.442486524581909
2.441765069961548
2.441054582595825
2.4403533935546875
2.439662218093872
2.4389805793762207
2.4383082389831543
2.437645435333252
2.436990976333618
2.4363460540771484
2.4357099533081055
2.435081958770752
2.

Bigram ile trigram loss'unu karşılaştır ve ürettiği isimlerin nasıl değiştiğini göster.

In [238]:
for i in range(5):
  out = []
  idx1 = 0
  idx2 = 0
  while True:
    xenc1 = F.one_hot(torch.tensor([idx1]), num_classes=27).float()
    xenc2 = F.one_hot(torch.tensor([idx2]), num_classes=27).float()
    xenc = torch.cat([xenc1, xenc2], dim=1)  
    logits = xenc @ W
    counts = logits.exp() 
    p = counts / counts.sum(1, keepdims=True)
    
    idx3= torch.multinomial(p, num_samples=1, replacement=True, generator=generator).item()
    out.append(itos[idx3])
    if idx3 == 0:
      break
    idx1 = idx2
    idx2 = idx3
  print(''.join(out))

tryahdachen.
ena.
da.
amiiah.
amkeles.
